# JAXperiments: HMC Sampler Demonstration

This notebook demonstrates a simple Hamiltonian Monte Carlo (HMC) sampler implemented in JAX for linear regression inference.

## Objective

We'll sample from the posterior distribution of linear regression parameters and compare:
- **Fixed seed sampling**: Using the same random seed repeatedly (shows autocorrelation issues)
- **Random seed sampling**: Using different random seeds (proper MCMC behavior)

The fixed seed approach demonstrates why proper random number generation is crucial for MCMC methods.

## Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path to import hmc package
notebook_dir = Path().absolute()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root))

import jax.numpy as jnp
import jax.random as random
import matplotlib.pyplot as plt
import numpy as np

from hmc.sampler import hmc_sample, log_posterior
from hmc.utils import generate_regression_data, get_hmc_config, plot_trace

# Set matplotlib style for better looking plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Generate Synthetic Regression Data

We'll generate data from a linear model: $y = mx + b + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2)$

In [ ]:
# Set true parameters for data generation
m_true = 2.5
b_true = 1.0
sigma_true = 0.5
n_data = 1500

# Generate synthetic data
data_key = random.PRNGKey(42)
x, y = generate_regression_data(data_key, n_data, m_true, b_true, sigma_true)

# Display the true parameters
print(f"True parameters:")
print(f"  Slope (m): {m_true}")
print(f"  Intercept (b): {b_true}")
print(f"  Noise σ: {sigma_true}")
print(f"  log(σ): {np.log(sigma_true):.4f}")
print(f"\nGenerated {n_data} data points")

### Visualize the Generated Data

In [ ]:
# Create scatter plot of the data
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(x, y, alpha=0.3, s=20, label='Data')

# Plot true regression line
x_line = np.linspace(x.min(), x.max(), 100)
y_line = m_true * x_line + b_true
ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'True line: y = {m_true}x + {b_true}')

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Synthetic Linear Regression Data', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()